# Running OGGM with the daily temperature-index mass-balance model

By the end, users should understand how to:

- use OGGM’s daily mass-balance model from the development branch;
- process daily W5E5 climate data;
- recalibrate the daily model using the existing monthly calibration as a starting point (this is only possible because we are using the same basline climate `W5E5` but at a different resolution;
- run glacier evolution with daily hydro diagnostics;
- inspect daily runoff and melt variables.

This is an advanced tutorial because it does not simply use the default monthly OGGM workflow. It modifies the mass-balance model, climate input, calibration suffixes, and diagnostic outputs.

This tutorial is not showing you sensitivity experiments or how to investigate uncertainty. For that referred to:

- [Runoff sensitivity tutorial](https://tutorials.oggm.org/stable/notebooks/tutorials/runoff_sensitivity.html)

## Stage 1 — Environment setup: use OGGM dev branch

```bash
cd ~/oggm
git status
git fetch origin
git checkout -b dev origin/dev
git status
pip install -e .
```
This tutorial requires the OGGM development branch because the daily model functionality is not necessarily available in the stable release used by beginner tutorials. The commands above shwo you how to checkout the dev branch. For this you need to have clone the OGGM repository. 

Follow this instructions if you havent done that:
 - [Install OGGM from source](https://docs.oggm.org/en/stable/installing-oggm.html#from-source-editable)

## Stage 2 — Imports and configuration file

This are all the required imports

In [ ]:
# Module logger
import logging
import sys
import configparser
import geopandas as gpd
import xarray as xr
from functools import partial
import numpy as np

from oggm import cfg, utils, workflow, tasks, DEFAULT_BASE_URL
from oggm.exceptions import InvalidParamsError, InvalidWorkflowError
from oggm.shop.w5e5 import process_w5e5_data
from oggm.sandbox import distribute_2d
from oggm.core import massbalance
from oggm.core.flowline import FileModel
from oggm import entity_task
from oggm.utils import float_years_timeseries

In [ ]:
log = logging.getLogger(__name__)

In [ ]:
cfg.initialize(logging_level='DEBUG')
cfg.PATHS['working_dir'] = utils.gettempdir(dirname='OGGM-daily', reset=True)

We need the following configuration for OGGM that will allow us to explore all files in the glacier directory

In [ ]:
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_compression'] = True
cfg.PARAMS['use_tar_shapefiles'] = True
cfg.PARAMS['use_temp_bias_from_file'] = True
cfg.PARAMS['compress_climate_netcdf'] = False
cfg.PARAMS['store_model_geometry'] = True
cfg.PARAMS['store_fl_diagnostics'] = True


cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes']        = 8
cfg.PARAMS['border']              = 80

## Stage 3 — Select glaciers within a river catchment in the Alps

We will focus on a group of glaciers within the catchment of a river in the Alps. For that I have computed a specific catchment and selected glaciers within that catchment area which is provided in a vector format in the data directory of this tutorial.

In [ ]:
from pathlib import Path

DATA_DIR = Path.cwd().resolve().parents[1] / "Lanzhou_workshop_oggm_tutorials/data"
print(DATA_DIR)

catchment_path = DATA_DIR / 'rofental_river_catchment/deims_sites_boundariesPolygon.shp'
print(catchment_path)

fr = utils.get_rgi_region_file(11, version='62', reset=False)
gdf = gpd.read_file(fr)

rof_shp = gpd.read_file(catchment_path)
rof_sel = gdf.clip(rof_shp)
rof_sel = rof_sel.sort_values('Area', ascending=False)


In [ ]:
rof_sel.plot()

## Stage 4 — Initialize glacier directories from preprocessing level 4

In this tutorial, we are not changing the glacier outlines, topography or other observations used for calibration. The main change is the temporal resolution of the baseline climate data: we move from monthly to daily climate forcing while still using `W5E5` as the reanalysis product.

Because the glacier geometry and inversion setup are unchanged, we do not need to repeat preprocessing levels 0 to 4. Instead, we start from OGGM preprocessed glacier directories at level 4, using the default OGGM preprocessing URL.

However, the daily mass-balance model uses a different climate file with higher temporal resolution. Therefore, even though the glacier directories already contain a standard calibration, we need to recalibrate the mass-balance model for the daily climate data before running the simulation. Below I show one way of doing this.

In [ ]:
base_url = ('https://cluster.klima.uni-bremen.de/~oggm/'
                'gdirs/oggm_v1.6/L3-L5_files/2025.6/elev_bands/W5E5/per_glacier_spinup')

gdirs = workflow.init_glacier_directories(
    rof_sel,
    from_prepro_level=4,
    prepro_base_url=base_url,
    reset=True,
    force=True
)

## Stage 5 — Introducion of the daily-model

In the code below we set up a especific OGGM configuration that will give you flexibility and the option to re-calibrate the MB model.

- The `check_calib_params=False` is used here because the tutorial recalibrates the daily model starting from existing calibration information.

In [ ]:
DailyTI_nocheck = partial(massbalance.DailyTIModel, check_calib_params=False)

MBModel = DailyTI_nocheck

settings_filesuffix = '_daily'
observations_filesuffix = '_daily'

climate_filename = 'climate_historical_daily'

cfg.PARAMS['baseline_climate'] = 'GSWP3_W5E5_daily'

### Get daily climate data

In [ ]:
workflow.execute_entity_task(process_w5e5_data, gdirs, daily=True);

### Re-calibrate the daily MB model

The preprocessed glacier directories already contain a standard OGGM mass-balance calibration. Here, we use this calibration as the starting point for **re-adjusting** the calibration to the daily model.

We can do this because we are not changing the glacier geometry or the climate product: we still use `W5E5`, and the same glacier outlines, elevation-band flowlines, and ice-thickness inversion remain the same. The main change is the temporal resolution of the mass-balance model, *from monthly to daily*.

For this reason, we keep the original:

- `prcp_fac`, which corrects precipitation biases in the climate data;
- `temp_bias`, which corrects systematic temperature offsets.

We do not recalibrate these parameters here because they describe the climate forcing itself. Recalibrating `prcp_fac`, `temp_bias`, and `melt_f` together would add too much freedom: different parameter combinations could produce a similar annual mass balance.

Instead, we recalibrate only the melt factor, `melt_f`. This parameter is most affected by the change from monthly to daily data, because daily forcing resolves short warm periods that are smoothed in monthly averages.

In this tutorial, the calibration question is therefore:

- Keeping the same W5E5 climate corrections, what daily `melt_f` is needed to reproduce the reference mass balance?

In [ ]:
for gdir in gdirs:
    mb_calib = gdir.read_json('mb_calib')
    
    tasks.mb_calibration_from_scalar_mb(
            gdir,
            ref_mb=mb_calib['reference_mb'],
            ref_mb_err=mb_calib['reference_mb_err'],
            ref_mb_period=mb_calib['reference_period'],
            settings_filesuffix=settings_filesuffix,
            observations_filesuffix=observations_filesuffix,
            write_to_gdir=True,
            overwrite_gdir=True,
            overwrite_observations=True,
            calibrate_param1='melt_f',
            calibrate_param2=None,
            calibrate_param3=None,
            prcp_fac=mb_calib['prcp_fac'],
            temp_bias=mb_calib['temp_bias'],
            mb_model_class=MBModel,
            filename=climate_filename
        )

### Run the daily MB model

In [ ]:
# compute apparent MB
workflow.execute_entity_task(tasks.apparent_mb_from_any_mb,
                             gdirs,
                             mb_model_class=MBModel);

### Calibrate ice dynamic parameters to match a reference Volume

In [ ]:
# calibrate ice dynamic params
workflow.calibrate_inversion_from_consensus(
    gdirs,
    apply_fs_on_mismatch=True,
    error_on_mismatch=True,  # if you're running many glaciers some might not work
    filter_inversion_output=True,  # this partly filters the over deepening due to
    #    # the equilibrium assumption for retreating glaciers (see. Figure 5 of Maussion et al. 2019)
);

### Initialise present day glacier.

In [ ]:
workflow.execute_entity_task(tasks.init_present_time_glacier, gdirs);

## Stage 6 — Run the daily model with hydro diagnostics

In [ ]:
from daily_model_tools import run_with_hydro_daily

> **Important**
>
> `run_with_hydro_daily` is not currently a standard OGGM task. It was adapted for this tutorial from the [OGGM mass-balance sandbox](https://github.com/OGGM/massbalance-sandbox) and related daily mass-balance/runoff work. We use it here as a prototype to explore daily hydro diagnostics.
>
> Future OGGM versions may include similar functionality, but the official implementation may compute or store runoff variables differently.

In [ ]:
cfg.PARAMS['store_model_geometry'] = True

y0 = 1980
y1 = 2019

simulation_name = '_climate_historical_daily'

workflow.execute_entity_task(
            run_with_hydro_daily,
            gdirs,
            settings_filesuffix=settings_filesuffix,
            run_task=tasks.run_from_climate_data,
            ys=y0, ye=y1,
            climate_filename=climate_filename,
            climate_input_filesuffix='',
            mb_model_class=DailyTI_nocheck,
            output_filesuffix=simulation_name,
            mb_elev_feedback='annual',
            store_monthly_step=False,
            store_annual=True,
        );

## Projected thickness

In [ ]:
workflow.execute_entity_task(distribute_2d.add_smoothed_glacier_topo, gdirs)
workflow.execute_entity_task(distribute_2d.assign_points_to_band, gdirs)
workflow.execute_entity_task(distribute_2d.distribute_thickness_from_simulation,
                             gdirs, input_filesuffix=simulation_name)

In [ ]:
import os
os.listdir(gdirs[0].dir)

## Visualise runoff for all 33 glaciers

In [ ]:
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

runoff_vars = [
    'melt_off_glacier',
    'melt_on_glacier',
    'liq_prcp_off_glacier',
    'liq_prcp_on_glacier'
]

rows = []

for gdir in gdirs:
    # This assumes the file suffix is "_climate_historical_daily"
    # i.e. model_diagnostics_climate_historical_daily.nc
    fpath = gdir.get_filepath(
        'model_diagnostics',
        filesuffix='_climate_historical_daily'
    )

    with xr.open_dataset(fpath) as ds:
        # Sum over all available years
        comp = {}
        for var in runoff_vars:
            comp[var] = float(ds[var].sum(skipna=True))

        total_runoff = sum(comp.values())

        row = {
            'rgi_id': gdir.rgi_id,
            'name': getattr(gdir, 'name', gdir.rgi_id),
            'area_km2': gdir.rgi_area_km2,
            'total_runoff_kg': total_runoff,
        }

        row.update(comp)
        rows.append(row)

df = pd.DataFrame(rows)
df

In [ ]:
df_frac = df.copy()

for var in runoff_vars:
    df_frac[var] = df[var] / df['total_runoff_kg']

df_frac

In [ ]:
# Sort glaciers by area, largest to smallest
df_plot = df_frac.sort_values('area_km2', ascending=False).copy()

# Short labels for readability
df_plot['label'] = [f'G{i+1}' for i in range(len(df_plot))]

plot_df = df_plot.set_index('label')[runoff_vars]

# Use RGI ID on the bottom x-axis
plot_df = df_plot.set_index('rgi_id')[runoff_vars]

import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 5))

plot_df.plot(
    kind='bar',
    stacked=True,
    ax=ax
)

ax.set_ylabel('Fraction of total runoff')
ax.set_xlabel('Glacier RGI ID, ordered by area')
ax.set_title('Relative runoff contribution by glacier over simulation period')

# Rotate RGI IDs for readability
ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=45,
    ha='right'
)

ax.legend(
    title='Runoff component',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

# Add glacier area as a second x-axis
ax_top = ax.secondary_xaxis('top')
ax_top.set_xticks(np.arange(len(df_plot)))
ax_top.set_xticklabels(
    [f"{a:.1f}" for a in df_plot['area_km2']],
    rotation=45,
    ha='left'
)
ax_top.set_xlabel('Glacier area (km²)')

plt.tight_layout()
plt.show()

In [ ]:
annual_rows = []

for gdir in gdirs:
    fpath = gdir.get_filepath(
        'model_diagnostics',
        filesuffix='_climate_historical_daily'
    )

    with xr.open_dataset(fpath) as ds:
        for yr in ds.time.values[:-1]:  # last time often has NaNs
            row = {
                'rgi_id': gdir.rgi_id,
                'year': int(yr),
                'area_km2': gdir.rgi_area_km2,
            }

            for var in runoff_vars:
                row[var] = float(ds[var].sel(time=yr))

            row['total_runoff'] = sum(row[v] for v in runoff_vars)
            annual_rows.append(row)

df_annual = pd.DataFrame(annual_rows)

In [ ]:
catchment_ts = (
    df_annual
    .groupby('year')[runoff_vars]
    .sum()
)

ax = catchment_ts.plot(
    figsize=(12, 5)
)

ax.set_ylabel('Runoff contribution (kg yr⁻¹)')
ax.set_xlabel('Year')
ax.set_title('Catchment runoff components through time')
ax.legend(title='Runoff component', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
catchment_frac = catchment_ts.div(catchment_ts.sum(axis=1), axis=0)

ax = catchment_frac.plot(
    kind='area',
    stacked=True,
    figsize=(12, 5)
)

ax.set_ylabel('Fraction of total runoff')
ax.set_xlabel('Year')
ax.set_title('Catchment runoff composition through time')
ax.legend(title='Runoff component', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

## What about future simulations?

In this tutorial, we use daily `W5E5` to demonstrate the daily mass-balance model and to compute daily hydro diagnostics for the historical period. However, most future climate scenarios used in OGGM are available as monthly bias-corrected climate files, not daily files.

Therefore, we do not extend `run_with_hydro_daily` to future projections here. Instead, the recommended workflow is:

- use the daily model for historical experiments and process understanding;
- use the standard monthly OGGM workflow for future projections;
- compare daily and monthly historical simulations to estimate how much the temporal resolution affects melt, snowfall, liquid precipitation, and runoff diagnostics in your study region.
